<a href="https://colab.research.google.com/github/AvanthaPeiris/n8n/blob/master/LanGraph_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain==1.5.0 -q
!pip install langchain.openai -q
!pip install langgraph -q
##!pip install -U langchain-google-genai langchain-core -q

In [2]:
!pip install -U langchain-google-genai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [3]:
from typing import TypedDict, List
from langchain_core.messages import HumanMessage

In [4]:
class agentState(TypedDict):
  msg: List[HumanMessage]

In [6]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('ChatGPT')

In [32]:
import os
import getpass
os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_new')

# #getpass.getpass("Enter your Gemini API key: ")


In [8]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

In [45]:
def chatBot(state:agentState) -> agentState:
  # Using with_retry on the runnable to automatically retry on transient API errors
  runnable_with_retry = llm.with_retry(
      stop_after_attempt=3,
      wait_exponential_jitter=True
  )
  response = runnable_with_retry.invoke(state['msg'])
  #response = model.invoke(state['msg'])
  print(f'\nAI: {response.content}')
  return state

In [46]:
from langgraph.graph import StateGraph,START,END

workflow=StateGraph(agentState)

workflow.add_node("chatbot",chatBot)

workflow.add_edge(START,"chatbot")
workflow.add_edge("chatbot",END)

app=workflow.compile()




In [49]:
user_prompt=input('Enter your query :')
results = app.invoke({"msg":[HumanMessage(content=user_prompt)]})
print(results)

Enter your query :What is Langgraph in simple term?


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AI: [{'type': 'text', 'text': 'In simple terms, **LangGraph** is a tool that lets you build **smart AI workflows that can loop, make decisions, and work together like a team.**\n\nThink of standard AI chatbots like a vending machine: you put in a prompt (money), and you get an answer (a snack). It’s a straight line.\n\n**LangGraph is more like a human worker:**\n1. It looks at your request.\n2. It tries to solve it.\n3. It checks its own work (*"Did I get this right?"*).\n4. If it\'s wrong, it tries again (a **loop**).\n5. It might ask another AI expert for help (multi-agent teamwork).\n\n### Why do we need it?\nStandard AI is great at answering in one go, but complex tasks require steps. For example: *"Write a research report on'}, {'type': 'text', 'text': ' electric cars."* \nAn AI alone might hallucinate or miss things. With LangGraph, you can program a loop:\n* **Step 1:** AI writes a draft.\n* **Step 2:** A second AI checks it for facts.\n* **Step 3:** If facts are wrong, send it

In [51]:
user_prompt=input('Enter your query :')
while user_prompt!='exit':
  results = app.invoke({"msg":[HumanMessage(content=user_prompt)]})
  print(results)
  user_prompt=input('Enter your query :')
print("Bye for now! see you soon!")

Enter your query :test

AI: [{'type': 'text', 'text': 'Test received loud and clear! How can I help you today?', 'extras': {'signature': 'EpYBCpMBAWkUfRNN41MvRYKFMZSkk/G9exeUNuf14RoNuXkxT0l0KIPvws6ulQtQS5eG0CindADKR1OESUD386vevRwqfX29e5ZpMcUfmj1UZRlwSGgSBHcQlz8RZwOZsGJCSDxrc8ENJFMvDdbg3as1kuvrGUPYDg5ufT5Gd9hvx7+MbXXjot3oE9RPcrarYqxiALT5ciVr'}}]
{'msg': [HumanMessage(content='test', additional_kwargs={}, response_metadata={})]}
Enter your query :help me

AI: [{'type': 'text', 'text': "I'm here for you. What's going on? Tell me how I can help.", 'extras': {'signature': 'EpYBCpMBAWkUfRPGlna4Bv6pdHfLC3QaoE0W25+wYyMxbixQeSbL5BzBOvEBKShZibUkK8pTwx+0AyjCpMsmzlxOJi5s2D1BxPfC/8qKbXUTj1lXAAsZwpomOxDYCs9Vs0qn7+O8xVT/Ntn5L+86w53HGZxbOf2dSeRgVOIH87q7rMKgVdXTfPK3Od5FoVMPyoaKgLOu+AwC'}}]
{'msg': [HumanMessage(content='help me', additional_kwargs={}, response_metadata={})]}
Enter your query :exit
Bye for now! see you soon!
